# Experiment 5: FinBERT encoder baseline

This experiment fine-tunes the complete pretrained `yiyanghkust/finbert-pretrain` encoder on the complete training set. It reports validation/test macro-F1 and saves the baseline metrics without hard-sample selection or targeted training.

## Imports and reproducibility

In [1]:
import copy
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertConfig, BertForSequenceClassification, BertTokenizer, get_linear_schedule_with_warmup

sys.path.append("..")
from utils.preprocessing import LABELS, MAX_LEN, encode_labels

HF_CACHE_DIR = os.path.abspath("../.cache/huggingface")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["HF_TOKEN_PATH"] = os.path.join(HF_CACHE_DIR, "token")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

SEED = 42
MODEL_NAME = "yiyanghkust/finbert-pretrain"
MAX_LENGTH = MAX_LEN
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
FINBERT_CONFIG = BertConfig(
    vocab_size=30873, hidden_size=768, num_hidden_layers=12,
    num_attention_heads=12, intermediate_size=3072, hidden_act="gelu",
    hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
    max_position_embeddings=512, type_vocab_size=2,
    num_labels=len(LABELS),
    id2label={index: label for index, label in enumerate(LABELS)},
    label2id={label: index for index, label in enumerate(LABELS)},
)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed()
print("Device:", DEVICE)


W1106 18:39:08.693000 25435 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Device: cpu


## Load data and tokenize

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
y_test = encode_labels(test_df.label)

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, cache_dir=HF_CACHE_DIR, token=False, use_fast=False,)

def tokenize(frame):
    encoded = tokenizer(
        frame.sentence.tolist(), padding="max_length", truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    return encoded["input_ids"], encoded["attention_mask"]

input_train, mask_train = tokenize(train_df)
input_val, mask_val = tokenize(val_df)
input_test, mask_test = tokenize(test_df)
print(f"model={MODEL_NAME} vocab={FINBERT_CONFIG.vocab_size}")
print(input_train.shape, input_val.shape, input_test.shape)


model=yiyanghkust/finbert-pretrain vocab=30873
torch.Size([2901, 64]) torch.Size([967, 64]) torch.Size([968, 64])


## Training and evaluation helpers

In [3]:
def loader(input_ids, attention_mask, labels, shuffle=False):
    tensors = [input_ids, attention_mask, torch.tensor(labels, dtype=torch.long)]
    return DataLoader(TensorDataset(*tensors), batch_size=BATCH_SIZE, shuffle=shuffle)


def new_model():
    set_seed()
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, config=FINBERT_CONFIG, cache_dir=HF_CACHE_DIR, token=False, ignore_mismatched_sizes=True,)
    return model.to(DEVICE)


@torch.no_grad()
def predict(model, input_ids, attention_mask, labels):
    model.eval(); probabilities=[]; predictions=[]; losses=[]
    for batch in loader(input_ids, attention_mask, labels):
        ids, masks, batch_labels = [value.to(DEVICE) for value in batch]
        output = model(input_ids=ids, attention_mask=masks, labels=batch_labels)
        probabilities.append(torch.softmax(output.logits, dim=1).cpu().numpy())
        predictions.append(output.logits.argmax(dim=1).cpu().numpy())
        losses.append(output.loss.item())
    return np.concatenate(probabilities), np.concatenate(predictions), float(np.mean(losses))


def fit_model(model, input_ids, attention_mask, labels, validation_data, epochs):
    train_loader = loader(input_ids, attention_mask, labels, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, max(1, steps // 10), steps)
    best_state = copy.deepcopy(model.state_dict()); best_score = -np.inf
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            ids, masks, batch_labels = [value.to(DEVICE) for value in batch]
            output = model(input_ids=ids, attention_mask=masks, labels=batch_labels)
            loss = output.loss
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        _, val_predictions, val_loss = predict(model, *validation_data)
        val_score = f1_score(validation_data[2], val_predictions, average="macro")
        print(f"epoch {epoch + 1}/{epochs} - val_loss: {val_loss:.4f} - val_macro_f1: {val_score:.4f}")
        if val_score > best_score:
            best_score = val_score; best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model


def score(model, input_ids, attention_mask, labels):
    probabilities, predictions, _ = predict(model, input_ids, attention_mask, labels)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
    }, probabilities, predictions


## Train the baseline transformer


In [4]:
validation_data = (input_val, mask_val, y_val)
baseline_transformer = fit_model(
    new_model(), input_train, mask_train, y_train, validation_data,
    epochs=EPOCHS,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

epoch 1/2 - val_loss: 0.4234 - val_macro_f1: 0.7946
epoch 2/2 - val_loss: 0.4060 - val_macro_f1: 0.8213


## Evaluate the baseline transformer


## Model size


In [5]:
total_parameters = sum(parameter.numel() for parameter in baseline_transformer.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in baseline_transformer.parameters()
    if parameter.requires_grad
)

print(f"Total parameters: {total_parameters:,} ({total_parameters / 1e6:.2f}M)")
print(f"Trainable parameters: {trainable_parameters:,} ({trainable_parameters / 1e6:.2f}M)")
print(f"Estimated FP32 weights: {total_parameters * 4 / 1024**2:.1f} MiB")
print(f"Estimated FP16 weights: {total_parameters * 2 / 1024**2:.1f} MiB")
print(f"Estimated INT8 weights: {total_parameters / 1024**2:.1f} MiB")


Total parameters: 109,754,115 (109.75M)
Trainable parameters: 109,754,115 (109.75M)
Estimated FP32 weights: 418.7 MiB
Estimated FP16 weights: 209.3 MiB
Estimated INT8 weights: 104.7 MiB


## Report baseline transformer statistics


In [6]:
validation_metrics, _, validation_predictions = score(baseline_transformer, input_val, mask_val, y_val)
test_metrics, _, test_predictions = score(baseline_transformer, input_test, mask_test, y_test)
results_df = pd.DataFrame([
    {"split": "validation", **validation_metrics},
    {"split": "test", **test_metrics},
])
print("Baseline FinBERT transformer statistics")
print(results_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nValidation classification report")
print(classification_report(y_val, validation_predictions, target_names=LABELS))
print("Test classification report")
print(classification_report(y_test, test_predictions, target_names=LABELS))


Baseline FinBERT transformer statistics
     split  accuracy  macro_f1
validation    0.8459    0.8213
      test    0.8678    0.8449

Validation classification report
              precision    recall  f1-score   support

    negative       0.79      0.79      0.79       121
     neutral       0.88      0.89      0.88       574
    positive       0.80      0.79      0.79       272

    accuracy                           0.85       967
   macro avg       0.82      0.82      0.82       967
weighted avg       0.85      0.85      0.85       967

Test classification report
              precision    recall  f1-score   support

    negative       0.76      0.86      0.81       121
     neutral       0.89      0.90      0.90       575
    positive       0.86      0.80      0.83       272

    accuracy                           0.87       968
   macro avg       0.84      0.85      0.84       968
weighted avg       0.87      0.87      0.87       968



## Save baseline transformer metrics and model


In [7]:
metrics = pd.read_csv("../results/metrics.csv")

baseline_transformer_row = {
    "condition": "baseline",
    "embedding_config": "finbert_pretrained",
    "architecture": "transformer_finbert",
    "accuracy_overall": test_metrics["accuracy"],
    "f1_overall": test_metrics["macro_f1"],
}

transformer_keys = ["condition", "embedding_config", "architecture"]
is_same_transformer = np.logical_and.reduce(
    [metrics[key] == baseline_transformer_row[key] for key in transformer_keys]
)
metrics = metrics.loc[~is_same_transformer]
metrics = pd.concat([metrics, pd.DataFrame([baseline_transformer_row])], ignore_index=True)
metrics.to_csv("../results/metrics.csv", index=False)

torch.save(
    {
        "state_dict": baseline_transformer.state_dict(),
        "config": FINBERT_CONFIG.to_dict(),
    },
    "../models/baseline_transformer.pt",
)


# Targeted training (RQ1)

Does giving the hard training sentences more weight help more than giving the same weight to random sentences?
This part uses the baseline trained above (`baseline_transformer`), so run the whole notebook from the top.

- **Strategy 1:** hard = lowest p% of the margin (top probability - second probability). A copy of the baseline is trained 1 more epoch with the hard sentences at weight 1.5. Control: the same number of random sentences at weight 1.5.
- **Strategy 2:** hard = top probability between 0.4 and 0.6. Only the hard sentences where all annotators agreed get weight 5. Every model is trained from scratch.

Takes about 2 hours on a Mac GPU.

In [ ]:
SEEDS = [42, 43, 44]
PERCENTILES = [10, 20, 30]

os.makedirs("../results/targeted_transformer", exist_ok=True)

# True if all annotators gave the same label (made in notebook 07)
agree_train = pd.read_csv("../results/hard_samples/agreement_train.csv").all_agree.values
agree_val = pd.read_csv("../results/hard_samples/agreement_val.csv").all_agree.values
print("all-agree sentences: train", agree_train.sum(), "| val", agree_val.sum())

## Training with weights

Same as `fit_model` above, but every sentence has a weight in the loss: loss = mean(weight * cross-entropy).
A sentence with weight 5 counts 5 times as much as a normal sentence.

In [ ]:
def fit_weighted(model, input_ids, attention_mask, labels, weights, epochs, lr):
    dataset = TensorDataset(input_ids, attention_mask, torch.tensor(labels, dtype=torch.long),
                            torch.tensor(weights, dtype=torch.float32))
    train_loader = DataLoader(dataset, batch_size=16, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, steps // 10, steps)

    best_score = -1
    for epoch in range(epochs):
        model.train()
        for ids, masks, batch_labels, batch_weights in train_loader:
            logits = model(input_ids=ids.to(DEVICE), attention_mask=masks.to(DEVICE)).logits
            losses = torch.nn.functional.cross_entropy(logits, batch_labels.to(DEVICE), reduction="none")
            loss = (losses * batch_weights.to(DEVICE)).mean()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        _, val_predictions, _ = predict(model, input_val, mask_val, y_val)
        val_score = f1_score(y_val, val_predictions, average="macro")
        print(f"   epoch {epoch + 1}/{epochs} - val_macro_f1: {val_score:.4f}")
        if val_score > best_score:
            best_score = val_score
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model


def score_subsets(probabilities, hard_val):
    # macro F1 on all val sentences, on the hard ones, and on all-agree / not all-agree
    predictions = probabilities.argmax(axis=1)
    scores = {}
    scores["f1_overall"] = f1_score(y_val, predictions, average="macro")
    scores["f1_hard"] = f1_score(y_val[hard_val], predictions[hard_val], average="macro")
    scores["f1_nonhard"] = f1_score(y_val[~hard_val], predictions[~hard_val], average="macro")
    scores["f1_allagree"] = f1_score(y_val[agree_val], predictions[agree_val], average="macro")
    scores["f1_not_allagree"] = f1_score(y_val[~agree_val], predictions[~agree_val], average="macro")
    scores["accuracy_overall"] = accuracy_score(y_val, predictions)
    return scores

## Hard sentences

To know which **train** sentences are hard we need predictions from a model that did not train on them,
so we split train into 5 parts and each part is predicted by a model trained on the other 4 (out-of-fold).
For **val** we use the baseline from above.

This takes ~15 min. If `transformer_band_train.csv` is already in `results/hard_samples` it is loaded instead.

In [ ]:
from sklearn.model_selection import StratifiedKFold

band_path = "../results/hard_samples/transformer_band_train.csv"

if os.path.exists(band_path):
    oof = pd.read_csv(band_path)[["p_negative", "p_neutral", "p_positive"]].values
    print("loaded", band_path)
else:
    oof = np.zeros((len(y_train), 3))
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for fold_train, fold_test in folds.split(train_df, y_train):
        print("fold")
        fold_model = fit_model(new_model(), input_train[fold_train], mask_train[fold_train], y_train[fold_train],
                               validation_data, epochs=EPOCHS)
        oof[fold_test], _, _ = predict(fold_model, input_train[fold_test], mask_train[fold_test], y_train[fold_test])
        del fold_model

    band = train_df[["sentence", "label"]].copy()
    band["p_negative"] = oof[:, 0]
    band["p_neutral"] = oof[:, 1]
    band["p_positive"] = oof[:, 2]
    band["max_prob"] = oof.max(axis=1)
    band["all_agree"] = agree_train
    band.to_csv(band_path, index=False)

val_probs, _, _ = predict(baseline_transformer, input_val, mask_val, y_val)
print("out-of-fold macro F1 on train:", round(f1_score(y_train, oof.argmax(axis=1), average="macro"), 4))

## Strategy 1: margin percentile

The margin cutoff is computed on train (lowest p%) and the same cutoff is used on val.

In [ ]:
sorted_oof = np.sort(oof, axis=1)
margin_train = sorted_oof[:, -1] - sorted_oof[:, -2]
sorted_val = np.sort(val_probs, axis=1)
margin_val = sorted_val[:, -1] - sorted_val[:, -2]

rows = []
for p in PERCENTILES:
    cutoff = np.percentile(margin_train, p)
    hard_idx = np.where(margin_train <= cutoff)[0]
    hard_val = margin_val <= cutoff
    print(f"P{p}: {len(hard_idx)} hard train sentences, {hard_val.sum()} hard val sentences")

    for seed in SEEDS:
        random_idx = np.random.default_rng(seed).choice(len(y_train), len(hard_idx), replace=False)
        for arm, idx in [("targeted", hard_idx), ("random", random_idx)]:
            weights = np.ones(len(y_train))
            weights[idx] = 1.5

            set_seed(seed)
            model = copy.deepcopy(baseline_transformer)   # copy, so the baseline itself does not change
            model = fit_weighted(model, input_train, mask_train, y_train, weights, epochs=1, lr=1e-5)
            probabilities, _, _ = predict(model, input_val, mask_val, y_val)
            del model

            row = {"arm": arm, "percentile": p, "seed": seed}
            row.update(score_subsets(probabilities, hard_val))
            rows.append(row)
            print(f"   {arm} seed {seed}: f1 {row['f1_overall']:.4f}, hard f1 {row['f1_hard']:.4f}")

strategy1 = pd.DataFrame(rows)
strategy1.to_csv("../results/targeted_transformer/strategy1.csv", index=False)

## Strategy 2: confidence band + clean labels

| arm | sentences with weight 5 |
|---|---|
| no_weight | none (reference) |
| clean_hard | hard and all-agree |
| random | same number, random from all of train |
| clean_random | same number, random from all-agree sentences that are not hard |

Check the number of clean-hard sentences first. If it is very small (< 30) the results do not mean much.

In [ ]:
hard_train = (oof.max(axis=1) >= 0.4) & (oof.max(axis=1) <= 0.6)
hard_val = (val_probs.max(axis=1) >= 0.4) & (val_probs.max(axis=1) <= 0.6)

clean_hard = np.where(hard_train & agree_train)[0]
clean_easy = np.where(~hard_train & agree_train)[0]
print("hard train:", hard_train.sum(), "| clean-hard:", len(clean_hard), "| hard val:", hard_val.sum())

rows = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    for arm in ["no_weight", "clean_hard", "random", "clean_random"]:
        weights = np.ones(len(y_train))
        if arm == "clean_hard":
            weights[clean_hard] = 5
        elif arm == "random":
            weights[rng.choice(len(y_train), len(clean_hard), replace=False)] = 5
        elif arm == "clean_random":
            n = min(len(clean_hard), len(clean_easy))   # only smaller than clean_hard on tiny test data
            weights[rng.choice(clean_easy, n, replace=False)] = 5

        print(arm, "seed", seed)
        model = new_model()
        set_seed(seed)
        model = fit_weighted(model, input_train, mask_train, y_train, weights, epochs=EPOCHS, lr=LEARNING_RATE)
        probabilities, _, _ = predict(model, input_val, mask_val, y_val)
        del model

        row = {"arm": arm, "seed": seed}
        row.update(score_subsets(probabilities, hard_val))
        rows.append(row)

strategy2 = pd.DataFrame(rows)
strategy2.to_csv("../results/targeted_transformer/strategy2.csv", index=False)

## Results

Mean and std over the seeds. Then targeted minus control for each seed (same seed = same random draws, so this is fairer).
A difference smaller than the std is not a real effect.

In [ ]:
columns = ["f1_overall", "f1_hard", "f1_nonhard", "f1_allagree", "f1_not_allagree"]
print("Strategy 1")
print(strategy1.groupby(["percentile", "arm"])[columns].agg(["mean", "std"]).round(3))
print("\nStrategy 2")
print(strategy2.groupby("arm")[columns].agg(["mean", "std"]).round(3))

In [ ]:
for p in PERCENTILES:
    targeted = strategy1[(strategy1.arm == "targeted") & (strategy1.percentile == p)].sort_values("seed")
    random_ = strategy1[(strategy1.arm == "random") & (strategy1.percentile == p)].sort_values("seed")
    diff = targeted.f1_overall.values - random_.f1_overall.values
    print(f"Strategy 1 P{p}: targeted - random = {diff.mean():+.4f} (std {diff.std(ddof=1):.4f})")

clean = strategy2[strategy2.arm == "clean_hard"].sort_values("seed").f1_overall.values
for control in ["random", "clean_random", "no_weight"]:
    other = strategy2[strategy2.arm == control].sort_values("seed").f1_overall.values
    diff = clean - other
    print(f"Strategy 2: clean_hard - {control} = {diff.mean():+.4f} (std {diff.std(ddof=1):.4f})")